<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase8-multi-agent-governance/02_kill_switch_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 8: Kill-Switch Architecture
**Goal**: Implement a kill-switch that halts the multi-agent
      audit system immediately on critical breach and
      requires human authorization to proceed.
**Regulatory mapping**: EU AI Act Article 14 human oversight
                    and system interruption.
**Date**: June 2026.
**Status**: In Progress

In [1]:
import time
import json
from datetime import datetime
from google import genai
from google.colab import userdata, drive
import os

# Setup
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# - SYSTEM UNDER AUDIT -
SYSTEM_UNDER_AUDIT = {
    "system_name":   "TalentMatch AI",
    "purpose":       "Automated CV screening and candidate ranking",
    "sector":        "employment",
    "Known_metrics": {
        "gender_disparate_impact":   0.43,
        "age_disparate_impact":      0.20,
        "ethnicity_disparate_impact":0.80,
    },
    "human_oversight": "None currently implemented",
    "documentation":   "Partial. No FRIA completed.",
}

# - AUDIT AGENTS -
def bias_audit_agent(system):
  metrics = system["Known_metrics"]
  findings = []
  critical = False
  for dimension, ratio in metrics.items():
    if ratio < 0.80:
      severity = "CRITICAL" if ratio < 0.50 else "HIGH"
      if severity == "CRITICAL":
        critical = True
      findings.append({"dimension": dimension, "ratio": ratio,
                        "severity": severity, "compliant": False})
    else:
      findings.append({"dimension": dimension, "ratio": ratio,
                        "severity": "PASS", "compliant": True})
  return {
      "agent":             "Bias Audit Agent",
      "article":           "EU AI Act Article 10",
      "findings":           findings,
      "critical_breach":    critical,
      "verdict":            "FAIL" if not all(
                            f["compliant"] for f in findings) else "PASS"
  }

def oversight_audit_agent(system):
  oversight =              system["human_oversight"].lower()
  has_oversight =          "none" not in oversight and oversight != ""
  return {
      "agent":             "Oversight Audit Agent",
      "article":           "EU AI Act Article 14",
      "findings":           system["human_oversight"],
      "critical_breach":    not has_oversight,
      "verdict":            "PASS" if has_oversight else "FAIL"
  }

def documentation_audit_agent(system):
  docs =         system["documentation"].lower()
  complete =     "partial" not in docs and "no " not in docs
  return {
      "agent":             "Documentation Audit Agent",
      "article":           "EU AI Act Article 11 and 18",
      "findings":           system["documentation"],
      "critical_breach":    False,
      "verdict":            "PASS" if complete else "FAIL"
  }

AUDIT_AGENTS = {
    "bias": bias_audit_agent,
    "oversight": oversight_audit_agent,
    "documentation": documentation_audit_agent,
}

print("====== KILL SWITCH SYSTEM READY ====== ")
print(f"System under audit: {SYSTEM_UNDER_AUDIT['system_name']}")
print(f"Audit agents:       {len(AUDIT_AGENTS)}")
print("\nReady to build kill-switch next ✅")


Mounted at /content/drive
====== KILL SWITCH SYSTEM READY ====== 
System under audit: TalentMatch AI
Audit agents:       3

Ready to build kill-switch next ✅


In [2]:
# - KILL-SWITCH STATE -
# The kill-switch is a system-wide state that, once triggered,
# halts all further agent execution.

class KillSwitch:
  """
  A system-wide circuit breaker. Once triggered,
  it halts all agents execution and requires explicit
  human authorization to reset.
  """
  def __init__(self):
    self.triggered         = False
    self.triggered_reason  = None
    self.trigger_agent     = None
    self.trigger_time      = None
    self.authorized_by     = None

  def trigger(self, agent, reason):
    """Activate the kill-switch. Halts the system."""
    self.triggered         = True
    self.triggered_reason  = reason
    self.trigger_agent     = agent
    self.trigger_time      = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    print(f"\n{'!' * 60}")
    print(f"KILL-SWITCH TRIGGERED")
    print(f"Agent: {agent}")
    print(f"Reason: {reason}")
    print(f"Time:   {self.trigger_time}")
    print(f"SYSTEM HALTED. Human authorization required to proceed.")
    print(f"{'!' * 60}\n")

  def reset(self, authorized_by):
    """Reset the kill-switch. Requires human authorization."""
    if not authorized_by:
      print("Reset denied, Human authorization required.")
      return False
    self.triggered         = False
    self.authorized_by     = authorized_by
    print(f"\nKill-switch authorized by: {authorized_by}")
    print("System may now proceed under human supervision.")
    return True

  def status(self):
    return "HALTED" if self.triggered else "OPERATIONAL"

# - GOVERNANCE CONTROLLER WITH KILL-SWITCH -
def governed_audit_with_killswitch(system, agents, kill_switch):
  """
  Runs audit agents with kill-switch protection.
  Halts IMMEDIATELY when a critical breach is detected.
  Does not complete remaining agents after a halt.
  """
  print("=" * 60)
  print(f"GOVERNED AUDIT: {system['system_name']}")
  print(f"kill-switch status: {kill_switch.status()}")
  print("=" * 60)

  audit_results = []
  agents_run    = 0
  agents_halted = 0

  for agent_name, agent_function in agents.items():

    # Check kill-switch before running each agent
    if kill_switch.triggered:
      print(f"\nSkipping {agent_name} agent. "
            f"kill-switch is active.")
      agents_halted += 1
      continue

    print(f"\nRunning {agent_name} audit agent...")
    result = agent_function(system)
    audit_results.append(result)
    agents_run += 1

    print(f"  Verdict: {result['verdict']}")

    # Trigger kill-switch on critical breach
    if result.get("critical_breach"):
      kill_switch.trigger(
          agent  = result["agent"],
          reason = f"Critical breach under {result['article']}"
      )

  print("\n" + "=" * 60)
  print("GOVERNED AUDIT SUMMARY")
  print("=" * 60)
  print(f"Agent run: {agents_run}")
  print(f"Agent halted: {agents_halted}")
  print(f"Kill-switch: {kill_switch.status()}")

  if kill_switch.triggered:
    print(f"\nAudit INCOMPLETE. System halted by kill-switch.")
    print(f"Triggered by: {kill_switch.trigger_agent}")
    print(f"Reason:       {kill_switch.triggered_reason}")
    print(f"Halt time:    {kill_switch.trigger_time}")
    print(f"\nHuman authorization required before audit can continue.")
    verdict = "HALTED BY KILL-SWITCH"
  else:
    verdict = "AUDIT COMPLETED"

  return {
      "system":        system["system_name"],
      "agents_run":    agents_run,
      "agents_halted": agents_halted,
      "kill_switch":   kill_switch.status(),
      "trigger_agent": kill_switch.trigger_agent,
      "triggered_reason":kill_switch.triggered_reason,
      "trigger_time":  kill_switch.trigger_time,
      "audit_results": audit_results,
      "verdict":       verdict,
  }

# - RUN THE AUDIT WITH KILL-SWITCH -
print("DEMONSTRATION 1: Audit with kill-switch active\n")

ks = KillSwitch()
result = governed_audit_with_killswitch(
    SYSTEM_UNDER_AUDIT, AUDIT_AGENTS, ks)

print(f"\nFinal verdict: {result['verdict']}")
print(f"Kill-switch status: {ks.status()}")

DEMONSTRATION 1: Audit with kill-switch active

GOVERNED AUDIT: TalentMatch AI
kill-switch status: OPERATIONAL

Running bias audit agent...
  Verdict: FAIL

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
KILL-SWITCH TRIGGERED
Agent: Bias Audit Agent
Reason: Critical breach under EU AI Act Article 10
Time:   2026-06-14 19:45:33
SYSTEM HALTED. Human authorization required to proceed.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Skipping oversight agent. kill-switch is active.

Skipping documentation agent. kill-switch is active.

GOVERNED AUDIT SUMMARY
Agent run: 1
Agent halted: 2
Kill-switch: HALTED

Audit INCOMPLETE. System halted by kill-switch.
Triggered by: Bias Audit Agent
Reason:       Critical breach under EU AI Act Article 10
Halt time:    2026-06-14 19:45:33

Human authorization required before audit can continue.

Final verdict: HALTED BY KILL-SWITCH
Kill-switch status: HALTED


In [5]:
# - DEMONSTRATION 2: Human authorization to reset -
# EU AI Act Article 14 requires that a human can
# intervene and authorize the system to proceed.

print("=" * 60)
print("DEMONSTRATION 2: Human Authorization Workflow")
print("=" * 60)

print(f"\nCurrent kill-switch status: {ks.status()}")
print(f"Triggered by: {ks.trigger_agent}")
print(f"Reason: {ks.triggered_reason}")

# Attempt reset WITHOUT authorization (should fail)
print("\n---Attempt 1: Reset without authorization---")
ks.reset(authorized_by=None)

# Attempt reset WITH human authorization (should succeed)
print("\n--- Attempt 2: Reset with named human authorization---")
ks.reset(authorized_by="Steve Onyeke, Lead AI Governance Auditor for Afrispan Data Labs")

print(f"\nKill-switch status after authorized reset: {ks.status()}")

# - DEMONSTRATION 3: Compliant system passes cleanly -
print("\n" + "=" * 60)
print("DEMONSTRATION 3: Compliant system passes cleanly (no kill-switch trigger)")
print("=" * 60)

COMPLIANT_SYSTEM = {
    "system_name":   "TalentMatch AI v2 (remediated)",
    "purpose":       "Automated CV screening with bias mitigation",
    "sector":        "employment",
    "Known_metrics": {
        "gender_disparate_impact":   0.85,
        "age_disparate_impact":      0.82,
        "ethnicity_disparate_impact":0.88,
    },
    "human_oversight": "Human review panel for all rejections",
    "documentation":   "Complete. FRIA completed and signed.",
}

ks2 = KillSwitch()
result2 = governed_audit_with_killswitch(
    COMPLIANT_SYSTEM, AUDIT_AGENTS, ks2)

print(f"\nFinal verdict: {result2['verdict']}")


DEMONSTRATION 2: Human Authorization Workflow

Current kill-switch status: OPERATIONAL
Triggered by: Bias Audit Agent
Reason: Critical breach under EU AI Act Article 10

---Attempt 1: Reset without authorization---
Reset denied, Human authorization required.

--- Attempt 2: Reset with named human authorization---

Kill-switch authorized by: Steve Onyeke, Lead AI Governance Auditor for Afrispan Data Labs
System may now proceed under human supervision.

Kill-switch status after authorized reset: OPERATIONAL

DEMONSTRATION 3: Compliant system passes cleanly (no kill-switch trigger)
GOVERNED AUDIT: TalentMatch AI v2 (remediated)
kill-switch status: OPERATIONAL

Running bias audit agent...
  Verdict: PASS

Running oversight audit agent...
  Verdict: PASS

Running documentation audit agent...
  Verdict: PASS

GOVERNED AUDIT SUMMARY
Agent run: 3
Agent halted: 0
Kill-switch: OPERATIONAL

Final verdict: AUDIT COMPLETED


## Findings: Kill-Switch Architecture

**System under audit:** TalentMatch AI
**Kill-switch:** Stateful circuit breaker class
**Demonstrations:** 3 scenarios
**Date:** June 2026
**Regulatory mapping:** EU AI Act Article 14 human
                        oversight and system interruption

### The Kill-Switch Class

| Method | Function |
|---|---|
| trigger() | Halts the system on critical breach |
| reset() | Resumes system, requires human authorization |
| status() | Reports OPERATIONAL or HALTED |

### Demonstration Results

| Demo | Scenario | Outcome |
|---|---|---|
| 1 | Critical breach (age DI 0.20) | Kill-switch halted system, 2 agents skipped |
| 2 | Reset without authorization | DENIED |
| 2 | Reset with named human | AUTHORIZED |
| 3 | Compliant system | All agents ran, no trigger, AUDIT COMPLETED |

### Key Findings

1. The kill-switch halted the system immediately on
   the first critical breach. Unlike Lesson 1 where
   all agents ran to completion, the kill-switch
   skipped the remaining two agents the moment the
   bias agent detected the age disparate impact of
   0.20. This is fail-fast governance.

2. Human authorization is mandatory for reset. The
   system rejected a reset attempt with no named
   authorizer and accepted only a reset with a named
   accountable human. The system cannot resume on its
   own under any circumstance. This directly satisfies
   EU AI Act Article 14 human oversight requirements.

3. The kill-switch does not produce false positives.
   The compliant system in Demonstration 3 passed all
   three agents cleanly with no kill-switch trigger.
   The kill-switch fires only on genuine critical
   breaches, not on every audit. This precision is
   essential. A kill-switch that halts everything is
   as useless as one that halts nothing.

4. Named authorization creates accountability. The
   reset required a named individual: "Steve Onyeke,
   Lead AI Governance Auditor." This connects directly
   to the FRIA accountability layer: a named human,
   a timestamp, and a documented authorization. The
   kill-switch reset is itself an accountable governance
   event.

### EU AI Act Article 14 Compliance

| Requirement | Implementation |
|---|---|
| System interruption capability | Kill-switch trigger() method |
| Human ability to intervene | reset() requires authorization |
| Prevent unsupervised operation | No self-reset possible |
| Proportionate to risk | Triggers only on critical breach |

### Connection to FRIA Accountability Layer

The named human authorization in the kill-switch reset
is the same accountability principle required in the
Phase 9 FRIA template. A named individual takes
documented responsibility for the decision to proceed.
The kill-switch reset log captures who authorized
continuation, when, and on what basis. This is the
tamper-evident accountability record that distinguishes
genuine governance from compliance theatre.

### Connection to Lesson 3
The kill-switch currently logs to screen. Lesson 3
adds immutable audit logging: every trigger, every
reset, every authorization captured in a tamper-evident
record. That log becomes the audit trail that proves
human oversight actually occurred.